# 7. Red neuronal (MLP) sobre rezagos escalados

In [1]:
%run functions.ipynb

In [2]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

df = load_data()

# El escalador se ajusta solo con el entrenamiento
scaler = MinMaxScaler().fit(df[["yt_true"]].iloc[:-24])
scaled = scaler.transform(df[["yt_true"]]).ravel()

P_MAX = 13


def mlp_forecast(seed):
    model = MLPRegressor(
        hidden_layer_sizes=(16,),
        learning_rate_init=0.01,
        max_iter=5000,
        random_state=seed,
    )
    forecast = recursive_forecast(model, scaled, P_MAX)
    valid = ~np.isnan(forecast)
    forecast[valid] = scaler.inverse_transform(forecast[valid].reshape(-1, 1)).ravel()
    return forecast


# Una sola red depende mucho de la semilla aleatoria
forecasts = np.array([mlp_forecast(seed) for seed in range(10)])
test_mae = np.abs(forecasts[:, -24:] - df.yt_true.values[-24:]).mean(axis=1)
pd.Series(test_mae.round(1), name="MAE prueba por semilla")

0     511.5
1     277.2
2     485.1
3    1017.7
4     738.7
5     314.3
6     287.0
7     718.7
8     480.3
9     318.0
Name: MAE prueba por semilla, dtype: float64

In [3]:
# Para no depender de la semilla se promedian las 10 redes

df["yt_pred_mlp"] = forecasts.mean(axis=0)

plot_time_series(df[["yt_true", "yt_pred_mlp"]])

<Figure size 1200x400 with 1 Axes>

In [4]:
metrics = compute_evaluation_metrics(df)
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_mlp
0,MSE Train,126590.20
1,MSE Test,354554.33
2,MAE Train,252.74
3,MAE Test,443.28


Con solo 228 meses de entrenamiento la red no supera al autorregresivo lineal, y su resultado varia mucho entre semillas.